In [14]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        logging.warning(f"CSV 파일이 없습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류: {csv_path}")
        return 0
    
    result = []

    for row in rows:
        name = row["이름"]
        number = row["학번"]

        mid = row["중간"].strip()
        fin = row["기말"].strip()
        hw = row["과제"].strip()

        if mid == "" or fin == "" or hw == "":
            logging.info(f"{name}: 평균 None, 등급 None 결측값 존재")
            result.append({
                "이름": name,
                "학번": number,
                "점수": {
                    "중간": int(mid) if mid else None,
                    "기말": int(fin) if fin else None,
                    "과제": int(hw) if hw else None,
                },
                "평균": None,
                "등급": None,
            })
        else:
            mid = int(mid)
            fin = int(fin)
            hw = int(hw)
            avg = mid * 0.3 + fin * 0.5 + hw * 0.2

            if avg >= 90:
                grade = "A"
            elif avg >= 80:
                grade = "B"
            elif avg >= 70:
                grade = "C"
            else:
                grade = "F"
            
            logging.info(f"{name}: 평균 {avg:.1f}, 등급 {grade}")
            result.append({
                "이름": name,
                "학번": number,
                "점수": {
                    "중간": mid,
                    "기말": fin,
                    "과제": hw,
                },
                "평균": avg,
                "등급": grade,
            })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return len(result)

make_report("scores.csv", "report.json")

with open("report.json", "r", encoding="utf-8") as f:
    print(f.read())

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None 결측값 존재


[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]


설명
결측값 처리 : 중간, 기말, 과제 중 하나라도 빈 칸이면 평균과 등급을 구할 수 없기 때문에 모두 None으로 처리
인코딩 : encoding을 명시하지 않으면 os 기본값에 따라 결과가 달라질 수 있어 명시했다.

In [17]:
class InvalidJamoError(ValueError):
    pass

def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {type(c)}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {repr(c)}")
    
    code = ord(c)
    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {repr(c)}")

inputs =  ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for input in inputs:
    try:
        result = classify_jamo(input)
        print(f"{repr(input)}: {result}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ': 자음
'ㅏ': 모음
'ㄲ': 자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이 아닙니다: 'AB'
[TypeError] str 타입이 아닙니다: <class 'int'>
'ㅎ': 자음
'ㅣ': 모음
[ValueError] 길이가 1이 아닙니다: ''


설명
InvalidJamoError를 ValueError 자식으로 만든 이유 : 한글 자모가 아니다 라는 것은 자료형은 맞지만 값이 조건에서 벗어난 경우이기 때문에 ValueError의 범주에 속한다. 따라서 ValueError의 자식으로 만드는 것이 적절하다 Exception을 직접 상속하면 너무 광범위해져서 호출자가 구별해 처리하기 어렵다
except 순서 : InvalidJamoError를 ValueError보다 먼저 잡았다. 부모 예외가 자식 예외도 함께 잡기 때문에 자식 클래스를 부모보다 뒤에 두게 된다면 부모 except에 걸려 구별이 불가능해진다